# Best Products Dashboard
Queries MySQL using SQLAlchemy, builds interactive charts with Plotly, and exports a self-contained HTML file.

In [3]:
# ── Cell 1: Imports & Connection ─────────────────────────────────────────────
import pandas as pd
from sqlalchemy import create_engine

# Update with your credentials
engine = create_engine('mysql+pymysql://root@localhost/ecommercedatabase')

print('Connected successfully')

Connected successfully


In [5]:
# ── Cell 2: Queries → DataFrames ─────────────────────────────────────────────

# Revenue, orders, and avg order value by category
df_categories = pd.read_sql("""
    SELECT
        category,
        COUNT(*)                         AS order_count,
        SUM(item_count)                  AS units_sold,
        ROUND(SUM(total_amount), 2)      AS total_revenue,
        ROUND(AVG(total_amount), 2)      AS avg_order_value
    FROM purchase_history
    GROUP BY category
    ORDER BY total_revenue DESC
""", engine)

# Category performance by customer segment
df_cat_seg = pd.read_sql("""
    SELECT
        ph.category,
        cs.segment_name,
        ROUND(SUM(ph.total_amount), 2) AS revenue
    FROM purchase_history ph
    JOIN customer_profiles cp ON ph.user_id = cp.user_id
    JOIN customer_segments cs ON cp.segment_id = cs.segment_id
    GROUP BY ph.category, cs.segment_name
    ORDER BY ph.category, revenue DESC
""", engine)

# Monthly sales by category (trend)
df_cat_monthly = pd.read_sql("""
    SELECT
        DATE_FORMAT(order_date, '%%Y-%%m') AS month,
        category,
        ROUND(SUM(total_amount), 2)      AS revenue
    FROM purchase_history
    GROUP BY month, category
    ORDER BY month
""", engine)

print('Queries complete')
print(df_categories)

Queries complete
      category  order_count  units_sold  total_revenue  avg_order_value
0  Electronics           12        22.0        3291.99           274.33
1     Clothing           11        15.0        1297.50           117.95
2    Groceries           10        20.0        1238.00           123.80
3       Beauty            7         9.0         853.99           122.00
4   Home Decor            6         8.0         751.99           125.33


In [7]:
# ── Cell 3: Build Charts & Export HTML ───────────────────────────────────────
import plotly.express as px

colors = ['#2563EB', '#16A34A', '#CA8A04', '#DC2626', '#9333EA', '#0891B2']

# Chart 1 — Total revenue by category (bar)
fig1 = px.bar(
    df_categories, x='category', y='total_revenue',
    title='Total Revenue by Category',
    labels={'category': 'Category', 'total_revenue': 'Revenue ($)'},
    color='category', color_discrete_sequence=colors,
    text='total_revenue'
)
fig1.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig1.update_layout(showlegend=False)

# Chart 2 — Units sold by category (horizontal bar)
fig2 = px.bar(
    df_categories, x='units_sold', y='category',
    orientation='h',
    title='Units Sold by Category',
    labels={'category': 'Category', 'units_sold': 'Units Sold'},
    color='category', color_discrete_sequence=colors
)
fig2.update_layout(showlegend=False)
fig2.update_yaxes(categoryorder='total ascending')

# Chart 3 — Avg order value by category (bar)
fig3 = px.bar(
    df_categories, x='category', y='avg_order_value',
    title='Average Order Value by Category',
    labels={'category': 'Category', 'avg_order_value': 'Avg Order Value ($)'},
    color='category', color_discrete_sequence=colors,
    text='avg_order_value'
)
fig3.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig3.update_layout(showlegend=False)

# Chart 4 — Revenue by category over time (line)
fig4 = px.line(
    df_cat_monthly, x='month', y='revenue', color='category',
    title='Monthly Revenue Trend by Category',
    labels={'month': 'Month', 'revenue': 'Revenue ($)', 'category': 'Category'},
    color_discrete_sequence=colors,
    markers=True
)

# ── Combine into one HTML file ───────────────────────────────────────────────
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset='utf-8'>
    <title>Best Products Dashboard</title>
    <style>
        body {{ font-family: Arial, sans-serif; background: #f8fafc; margin: 0; padding: 20px; }}
        h1 {{ color: #1e293b; border-bottom: 3px solid #2563EB; padding-bottom: 10px; }}
        p {{ color: #475569; font-size: 15px; }}
        .grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-top: 20px; }}
        .card {{ background: white; border-radius: 8px; padding: 10px; box-shadow: 0 1px 4px rgba(0,0,0,0.1); }}
        .full {{ grid-column: span 2; }}
    </style>
</head>
<body>
    <h1>Best Products Dashboard</h1>
    <p>Ranks product categories by revenue, units sold, and average order value. Shows monthly sales trends per category. Built with MySQL + Python.</p>
    <div class='grid'>
        <div class='card'>{fig1.to_html(full_html=False, include_plotlyjs='cdn')}</div>
        <div class='card'>{fig2.to_html(full_html=False, include_plotlyjs=False)}</div>
        <div class='card'>{fig3.to_html(full_html=False, include_plotlyjs=False)}</div>
        <div class='card full'>{fig4.to_html(full_html=False, include_plotlyjs=False)}</div>
    </div>
</body>
</html>
"""

import os

output_path = os.path.expanduser('~/Downloads/products_dashboard.html')

with open(output_path, 'w') as f:
    f.write(html_content)

print(f'Dashboard saved → {output_path}')

Dashboard saved → /Users/chenyuzhai/Downloads/products_dashboard.html
